# 11 · UMAP plots of groups and features

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`plot_groups_and_features`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`plot_groups_and_feats`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/plot_groups_and_feats) → `sc-plot-groups-and-feats` |

The final QC figures: the UMAP coloured by each categorical **group** (sample, batch, cell type, clusters) and by each continuous **feature** (QC metrics, doublet and cell cycle scores). Groups or features missing from `obs` are skipped.

> The workflow's default `groups` list `leiden_res_1.0`, but the clustering step names that column `leiden_res_1.00`, so the workflow's plot silently drops it. sc-tools matches such Leiden keys to the formatted name and warns about any group or feature it cannot find. `wf_config.GROUPS` adds `C_scANVI` to the workflow's defaults.

**Inputs**
- `${cohort_id}.final.h5ad`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `plots/${cohort_id}.groups.umap.png`
- `plots/${cohort_id}.features.umap.png`

**Previous:** [10_artifact_metrics](10_artifact_metrics.ipynb) · **Next:** -

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Run `sc-plot-groups-and-feats`

In [ ]:
cfg.run(
    "sc-plot-groups-and-feats",
    "--adata-input", cfg.out("final.h5ad"),
    "--groups", ",".join(cfg.GROUPS),
    "--output-group-umap-plot-prefix", cfg.COHORT_ID,
    "--features", ",".join(cfg.FEATURES),
    "--output-feature-umap-plot-prefix", cfg.COHORT_ID,
)

scanpy prefixes saved embedding plots with the basis name; rename them as the WDL does.

In [ ]:
plots = cfg.RUN_DIR / "plots"
for kind in ["groups", "features"]:
    src = plots / f"umap{cfg.COHORT_ID}_{kind}_umap.png"
    dst = plots / f"{cfg.COHORT_ID}.{kind}.umap.png"
    if src.exists():
        src.replace(dst)
    display(Image(filename=str(dst), width=600))

## 2. Outputs of the series

These are the files the workflow publishes to `cohort_analysis/`, re-created in `cfg.RUN_DIR`.

In [ ]:
for p in sorted(cfg.RUN_DIR.rglob("*")):
    if p.is_file() and not p.parent.name.endswith("_model"):
        print(f"{p.stat().st_size / 1e6:10.1f} MB  {p.relative_to(cfg.RUN_DIR)}")

## 3. What to check next

From the workflow's user guide, before building on these results:

- Are QC metrics comparable across samples (notebooks 02–03)?
- Do cells group by cell type rather than by sample or batch after integration (UMAPs here, scib in notebook 10)?
- Do MapMyCells (`cell_type`) and scANVI (`C_scANVI`) agree, and are the expected cell types present in the expected proportions (notebook 09)?
- Are the cell types you care about present in enough samples and cells to compare groups?

If not, consider other filters (notebook 03) or another MapMyCells reference (notebook 04 → `sc-annotate` in notebook 09).

**Previous:** [10_artifact_metrics](10_artifact_metrics.ipynb)